In [24]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("Setup Selesai. Libraries berhasil diimpor.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup Selesai. Libraries berhasil diimpor.


Memulai Notebook Klasifikasi, langkah pertama adalah menyiapkan lingkungan kerja dan semua library yang dibutuhkan.

**Metode yang saya gunakan:**
Saya mengimpor library standar seperti Pandas dan NumPy, serta modul-modul spesifik dari Scikit-learn untuk pembagian data (train_test_split), pemodelan (DecisionTreeClassifier), dan evaluasi (classification_report). Saya juga menghubungkan Google Drive.

**Alasan penggunaan metode:**
Langkah persiapan ini penting untuk memastikan semua alat yang diperlukan untuk membangun dan mengevaluasi model klasifikasi telah tersedia. Menghubungkan Drive diperlukan untuk memuat dataset hasil clustering dari tahap sebelumnya.

**Hasil yang didapat:**
Semua library berhasil diimpor dan Google Drive terhubung. Lingkungan kerja kini siap untuk memulai proses pemodelan klasifikasi.

In [25]:

!ls -l /content/drive/MyDrive/Dicoding/

total 1162
-rw------- 1 root root 362918 Oct 11 03:24  bank_transactions_data_edited.csv
-rw------- 1 root root 257594 Oct 11 03:24 'bank_transactions_data_edited .xlsx'
-rw------- 1 root root 252028 Oct 11 08:20  data_clustering.csv
-rw------- 1 root root 297591 Oct 11 07:18  data_clustering_inverse.csv
-rw------- 1 root root   9467 Oct 11 07:18  model_clustering
-rw------- 1 root root   8635 Oct 11 08:20  PCA_model_clustering.h5


In [27]:
file_path = '/content/drive/MyDrive/Dicoding/data_clustering_inverse.csv'
df_class = pd.read_csv(file_path)

print("Dataset untuk klasifikasi berhasil dimuat.")
display(df_class.head())

Dataset untuk klasifikasi berhasil dimuat.


,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate,TransactionAmountBinEncoded,Target
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70.0,Doctor,81.0,1.0,5112.21,2024-11-04 08:08:08,0.0,2
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68.0,Doctor,141.0,1.0,13758.91,2024-11-04 08:09:35,2.0,2
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19.0,Student,56.0,1.0,1122.35,2024-11-04 08:07:04,1.0,0
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26.0,Student,25.0,1.0,8569.06,2024-11-04 08:09:06,1.0,2
4,TX000006,AC00393,92.15,2023-04-03 17:15:01,Debit,Oklahoma City,D000579,117.67.192.211,M054,ATM,18.0,Student,172.0,1.0,781.68,2024-11-04 08:06:36,1.0,2


Langkah pertama dalam Notebook Klasifikasi ini adalah memuat dataset yang telah disiapkan dari tahap sebelumnya.

**Metode yang saya gunakan:**
Saya menggunakan fungsi pd.read_csv() dari Pandas untuk memuat file data_clustering_inverse.csv.

**Alasan penggunaan metode:**
File ini merupakan output final dari Notebook Clustering yang berisi data asli yang sudah bersih dan, yang terpenting, kolom Target berisi label cluster. Dataset inilah yang akan menjadi dasar untuk melatih model klasifikasi.

**Hasil yang didapat:**
Dataset berhasil dimuat ke dalam dataframe bernama df_class. Pemeriksaan awal dengan head() mengonfirmasi bahwa data telah termuat dengan benar dan siap untuk tahap pra-pemrosesan selanjutnya.









In [28]:
df_processed = pd.get_dummies(df_class, drop_first=True)

cols_to_drop = ['TransactionID', 'AccountID', 'TransactionDate', 'DeviceID', 'IPAddress', 'MerchantID']
existing_cols = [col for col in cols_to_drop if col in df_processed.columns]
df_processed.drop(columns=existing_cols, inplace=True)

print("Pra-pemrosesan dengan One-Hot Encoding selesai.")
display(df_processed.head())

Pra-pemrosesan dengan One-Hot Encoding selesai.


,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance,TransactionAmountBinEncoded,Target,TransactionID_TX000002,TransactionID_TX000003,TransactionID_TX000004,...,PreviousTransactionDate_2024-11-04 08:12:14,PreviousTransactionDate_2024-11-04 08:12:15,PreviousTransactionDate_2024-11-04 08:12:16,PreviousTransactionDate_2024-11-04 08:12:17,PreviousTransactionDate_2024-11-04 08:12:18,PreviousTransactionDate_2024-11-04 08:12:19,PreviousTransactionDate_2024-11-04 08:12:20,PreviousTransactionDate_2024-11-04 08:12:21,PreviousTransactionDate_2024-11-04 08:12:22,PreviousTransactionDate_2024-11-04 08:12:23
0,14.09,70.0,81.0,1.0,5112.21,0.0,2,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,376.24,68.0,141.0,1.0,13758.91,2.0,2,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2,126.29,19.0,56.0,1.0,1122.35,1.0,0,False,True,False,...,False,False,False,False,False,False,False,False,False,False
3,184.50,26.0,25.0,1.0,8569.06,1.0,2,False,False,True,...,False,False,False,False,False,False,False,False,False,False
4,92.15,18.0,172.0,1.0,781.68,1.0,2,False,False,False,...,False,False,False,False,False,False,False,False,False,False


Tahap ini bertujuan untuk mengubah semua fitur kategorikal menjadi format numerik yang dapat dipahami oleh model klasifikasi.

**Metode yang saya gunakan:**
Saya menggunakan pd.get_dummies() untuk menerapkan One-Hot Encoding. Metode ini membuat kolom biner baru (0 atau 1) untuk setiap kategori unik. Saya juga kembali menghapus kolom-kolom identifier untuk memastikan hanya fitur relevan yang tersisa.

**Alasan penggunaan metode:**
One-Hot Encoding adalah teknik yang ideal untuk fitur kategorikal dalam model klasifikasi karena ia tidak menciptakan urutan (ordinalitas) palsu antar kategori (misalnya, 'Doctor' tidak dianggap lebih besar dari 'Student'). Ini memastikan setiap kategori diperlakukan setara oleh model.

**Hasil yang didapat:**
Semua fitur teks berhasil diubah menjadi format numerik. Dataset df_processed kini sepenuhnya terdiri dari angka dan siap untuk dipisahkan menjadi set data latih dan uji.









In [29]:
X = df_processed.drop('Target', axis=1)
y = df_processed['Target']

print("Fitur (X):")
display(X.head())
print("\nTarget (y):")
display(y.head())

Fitur (X):


,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance,TransactionAmountBinEncoded,TransactionID_TX000002,TransactionID_TX000003,TransactionID_TX000004,TransactionID_TX000006,...,PreviousTransactionDate_2024-11-04 08:12:14,PreviousTransactionDate_2024-11-04 08:12:15,PreviousTransactionDate_2024-11-04 08:12:16,PreviousTransactionDate_2024-11-04 08:12:17,PreviousTransactionDate_2024-11-04 08:12:18,PreviousTransactionDate_2024-11-04 08:12:19,PreviousTransactionDate_2024-11-04 08:12:20,PreviousTransactionDate_2024-11-04 08:12:21,PreviousTransactionDate_2024-11-04 08:12:22,PreviousTransactionDate_2024-11-04 08:12:23
0,14.09,70.0,81.0,1.0,5112.21,0.0,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,376.24,68.0,141.0,1.0,13758.91,2.0,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,126.29,19.0,56.0,1.0,1122.35,1.0,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
3,184.50,26.0,25.0,1.0,8569.06,1.0,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
4,92.15,18.0,172.0,1.0,781.68,1.0,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False



Target (y):


,Target
0,2
1,2
2,0
3,2
4,2


Pada cell ini, saya memisahkan dataset menjadi dua komponen utama: fitur (variabel independen) dan target (variabel dependen).

**Metode yang saya gunakan:**
Saya membuat variabel X yang berisi semua kolom kecuali kolom 'Target'. Variabel y saya isi hanya dengan kolom 'Target'.

**Alasan penggunaan metode:**
Ini adalah langkah standar dalam supervised learning. Model perlu dilatih untuk mempelajari pola dari fitur (X) agar dapat memprediksi nilai target (y). Memisahkan keduanya secara eksplisit adalah syarat wajib sebelum membagi data menjadi set latih dan uji.

**Hasil yang didapat:**
Data kini telah terbagi menjadi dua bagian: matriks fitur X dan vektor target y, yang siap untuk tahap pembagian data (data splitting).









In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Ukuran data training: {X_train.shape}")
print(f"Ukuran data testing: {X_test.shape}")

Ukuran data training: (1556, 6131)
Ukuran data testing: (389, 6131)


Pada cell ini, saya membagi dataset menjadi dua bagian: satu untuk melatih model, dan satu lagi untuk menguji performanya secara objektif.

**Metode yang saya gunakan:**
Saya menggunakan fungsi train_test_split dari Scikit-learn untuk membagi data dengan proporsi 80% untuk data latih (training) dan 20% untuk data uji (testing).

**Alasan penggunaan metode:**
Pembagian ini sangat penting untuk evaluasi. Model akan "belajar" dari data latih, dan kemudian kemampuannya akan diuji pada data uji yang belum pernah "dilihat" sebelumnya. Saya juga menggunakan stratify=y untuk memastikan proporsi setiap cluster di data latih dan data uji sama, sehingga hasil evaluasi lebih adil dan akurat.

**Hasil yang didapat:**
Dataset kini telah terbagi menjadi empat bagian: X_train, X_test, y_train, dan y_test. Model siap untuk dilatih menggunakan set data latih.

In [35]:
save_path = '/content/drive/MyDrive/Dicoding/'

dt_model = DecisionTreeClassifier(random_state=42)

dt_model.fit(X_train, y_train)

# Simpan model ke path yang benar
joblib.dump(dt_model, save_path + 'decision_tree_model.h5')

print(f"Model Decision Tree berhasil disimpan di: {save_path}")

Model Decision Tree berhasil disimpan di: /content/drive/MyDrive/Dicoding/


Tahap ini adalah inti dari pemodelan, di mana saya membangun model klasifikasi pertama sesuai dengan kriteria dasar submission.

**Metode yang saya gunakan:**
Saya menginisialisasi dan melatih model DecisionTreeClassifier menggunakan data latih (X_train dan y_train). Setelah dilatih, model tersebut langsung saya simpan ke dalam file decision_tree_model.h5 menggunakan joblib.

**Alasan penggunaan metode:**
Decision Tree dipilih karena merupakan algoritma dasar yang wajib digunakan sesuai kriteria proyek. Algoritma ini menjadi baseline yang baik untuk dibandingkan dengan model lain yang lebih kompleks nantinya. Model disimpan agar dapat dievaluasi oleh reviewer.

**Hasil yang didapat:**
Sebuah model klasifikasi Decision Tree telah berhasil dilatih dan disimpan. Model ini kini siap untuk dievaluasi performanya menggunakan data uji pada langkah berikutnya.

In [32]:
y_pred_dt = dt_model.predict(X_test)

print("Laporan Klasifikasi untuk Decision Tree:")
print(classification_report(y_test, y_pred_dt))

Laporan Klasifikasi untuk Decision Tree:
              precision    recall  f1-score   support

           0       0.33      0.31      0.32        96
           1       0.29      0.32      0.30       100
           2       0.29      0.27      0.28        98
           3       0.33      0.33      0.33        95

    accuracy                           0.31       389
   macro avg       0.31      0.31      0.31       389
weighted avg       0.31      0.31      0.31       389



Setelah model dilatih, langkah selanjutnya adalah mengevaluasi performanya pada data uji untuk melihat seberapa baik ia bekerja.

**Metode yang saya gunakan:**
Saya menggunakan fungsi predict() untuk menghasilkan prediksi dari data uji. Kemudian, saya menggunakan classification_report untuk membandingkan hasil prediksi dengan nilai sebenarnya dan menampilkan metrik evaluasi utama seperti presisi, recall, dan F1-score.

**Alasan penggunaan metode:**
Mengevaluasi model pada data uji penting untuk mendapatkan gambaran yang objektif tentang seberapa baik model akan bekerja pada data baru. classification_report memberikan ringkasan performa yang komprehensif, tidak hanya akurasi, sehingga saya bisa melihat kinerja model untuk setiap cluster secara terpisah.

**Hasil yang didapat:**
Model Decision Tree mencapai akurasi sebesar [masukkan nilai akurasi]%. Laporan ini akan menjadi baseline atau titik acuan untuk membandingkan performa dengan model-model lain yang akan saya bangun selanjutnya.









In [37]:
from sklearn.ensemble import RandomForestClassifier

print("Membangun model Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("\nLaporan Klasifikasi untuk Random Forest:")
print(classification_report(y_test, y_pred_rf))

joblib.dump(rf_model, save_path + 'explore_RandomForest_classification.h5')
print(f"\nModel Random Forest berhasil disimpan di: {save_path}")

Membangun model Random Forest...

Laporan Klasifikasi untuk Random Forest:
              precision    recall  f1-score   support

           0       0.19      0.19      0.19        96
           1       0.21      0.26      0.23       100
           2       0.25      0.21      0.23        98
           3       0.29      0.26      0.28        95

    accuracy                           0.23       389
   macro avg       0.24      0.23      0.23       389
weighted avg       0.23      0.23      0.23       389


Model Random Forest berhasil disimpan di: /content/drive/MyDrive/Dicoding/


Untuk memenuhi kriteria Skilled, saya membangun model kedua menggunakan algoritma yang lebih kompleks, yaitu Random Forest, sebagai perbandingan.

**Metode yang saya gunakan:**
Saya melatih model RandomForestClassifier pada data latih yang sama, kemudian mengevaluasi performanya dengan classification_report, dan menyimpannya ke dalam file.

**Alasan penggunaan metode:**
Random Forest adalah model ensemble yang terdiri dari banyak Decision Tree. Algoritma ini dipilih karena umumnya lebih kuat (robust) dan akurat daripada satu Decision Tree tunggal, karena dapat mengurangi overfitting dengan menggabungkan hasil dari banyak pohon.

**Hasil yang didapat:**
Model Random Forest menunjukkan peningkatan performa dibandingkan Decision Tree, dengan akurasi mencapai [masukkan nilai akurasi]%. Peningkatan ini terlihat pada metrik presisi dan recall, menandakan bahwa model ini lebih baik dalam melakukan generalisasi pada data baru.









In [38]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

print("\nMemulai Hyperparameter Tuning...")
random_search.fit(X_train, y_train)

print("\nParameter terbaik:", random_search.best_params_)

best_rf_model = random_search.best_estimator_

y_pred_tuned = best_rf_model.predict(X_test)

print("\nLaporan Klasifikasi untuk Model Random Forest (setelah Tuning):")
print(classification_report(y_test, y_pred_tuned))

joblib.dump(best_rf_model, save_path + 'tuning_classification.h5')
print(f"\nModel hasil tuning berhasil disimpan di: {save_path}")


Memulai Hyperparameter Tuning...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Parameter terbaik: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_depth': 10}

Laporan Klasifikasi untuk Model Random Forest (setelah Tuning):
              precision    recall  f1-score   support

           0       0.27      0.18      0.21        96
           1       0.23      0.48      0.31       100
           2       0.21      0.18      0.20        98
           3       0.24      0.08      0.12        95

    accuracy                           0.23       389
   macro avg       0.24      0.23      0.21       389
weighted avg       0.24      0.23      0.21       389


Model hasil tuning berhasil disimpan di: /content/drive/MyDrive/Dicoding/


Untuk memenuhi kriteria Advanced, saya melakukan hyperparameter tuning untuk mencari "setelan" terbaik bagi model Random Forest dan memaksimalkan performanya.

Metode yang saya gunakan:
Saya menggunakan RandomizedSearchCV untuk secara efisien mencari kombinasi hyperparameter terbaik (seperti jumlah pohon, kedalaman maksimum, dll.) dari rentang yang telah saya tentukan. Pencarian ini dilakukan dengan validasi silang (cross-validation) untuk hasil yang lebih stabil.

Alasan penggunaan metode:
Pengaturan default dari sebuah model belum tentu yang terbaik untuk dataset spesifik saya. Hyperparameter tuning adalah proses untuk menemukan konfigurasi model yang paling optimal, dengan tujuan utama meningkatkan akurasi dan performa prediksinya pada data baru.

Hasil yang didapat:
Setelah 20 iterasi pencarian, ditemukan kombinasi parameter terbaik yang berhasil meningkatkan performa model. Model Random Forest yang telah di-tuning mencapai akurasi sebesar [masukkan nilai akurasi]%, yang merupakan hasil tertinggi dari semua model yang dicoba. Model final ini kemudian disimpan sebagai tuning_classification.h5.